# 📊 Projeto: Previsão de Demanda de Vendas - Rossmann Store Sales

## Notebook 01 - Exploração e Preparação dos Dados

### 🎯 Objetivo do Projeto
Desenvolver modelos preditivos para prever vendas diárias de lojas Rossmann, comparando diferentes técnicas de machine learning e identificando os principais fatores que influenciam o desempenho comercial.

### 📋 Estrutura deste Notebook
1. **Carregamento e Análise Inicial** - Entender a estrutura dos dados
2. **Análise Exploratória (EDA)** - Visualizar padrões e tendências
3. **Feature Engineering** - Criar variáveis preditivas
4. **Preparação para Modelagem** - Tratamento e split dos dados

---

In [ ]:
# ============================================================
# 1️⃣ IMPORTAÇÃO DE BIBLIOTECAS
# ============================================================
# Importamos todas as bibliotecas necessárias para análise e visualização

# Manipulação de dados
import pandas as pd
import numpy as np

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Configurações visuais
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# Warnings
import warnings
warnings.filterwarnings('ignore')

print("✅ Bibliotecas carregadas com sucesso!")

In [ ]:
# ============================================================
# 2️⃣ CARREGAMENTO DOS DADOS
# ============================================================
# Carregamos 3 datasets:
# - train.csv: vendas históricas (2013-2015)
# - test.csv: período para previsão (sem vendas)
# - store.csv: características das lojas

print("📂 Carregando datasets...\n")

train = pd.read_csv("../data/raw/train.csv", dtype={"StateHoliday": "string"})
test = pd.read_csv("../data/raw/test.csv")
store = pd.read_csv("../data/raw/store.csv")

print(f"📊 Train shape: {train.shape} (linhas, colunas)")
print(f"📊 Test shape:  {test.shape}")
print(f"📊 Store shape: {store.shape}")
print(f"\n🏪 Total de lojas únicas: {train['Store'].nunique()}")
print(f"📅 Período dos dados: {train['Date'].min()} até {train['Date'].max()}")

In [ ]:
# ============================================================
# 3️⃣ VISUALIZAÇÃO INICIAL DOS DADOS
# ============================================================
# Analisamos as primeiras linhas para entender a estrutura

print("🔍 Primeiras linhas do dataset de vendas:\n")
display(train.head(10))

print("\n🏪 Informações das lojas:\n")
display(store.head())

In [ ]:
# ============================================================
# 4️⃣ ANÁLISE DE TIPOS E VALORES AUSENTES
# ============================================================
# Identificamos tipos de dados e presença de valores nulos

print("📋 INFORMAÇÕES DO DATASET DE VENDAS")
print("="*60)
train.info()

print("\n📋 VALORES AUSENTES")
print("="*60)

# Função para análise de valores ausentes
def missing_analysis(df, name):
    missing = pd.DataFrame({
        'Coluna': df.columns,
        'Nulos': df.isnull().sum(),
        'Percentual': (df.isnull().sum() / len(df) * 100).round(2)
    }).sort_values('Nulos', ascending=False)
    
    missing = missing[missing['Nulos'] > 0]
    
    if len(missing) == 0:
        print(f"✅ {name}: Nenhum valor ausente encontrado!")
    else:
        print(f"⚠️ {name}: {len(missing)} colunas com valores ausentes\n")
        display(missing)
    
    return missing

missing_train = missing_analysis(train, "TRAIN")
print("\n" + "="*60 + "\n")
missing_store = missing_analysis(store, "STORE")

In [ ]:
# ============================================================
# 5️⃣ ESTATÍSTICAS DESCRITIVAS
# ============================================================
# Analisamos distribuições, médias e valores extremos

print("📊 ESTATÍSTICAS DESCRITIVAS - VARIÁVEIS NUMÉRICAS\n")
display(train.describe())

print("\n💡 INSIGHTS INICIAIS:")
print("="*60)
print(f"📈 Vendas médias diárias: €{train['Sales'].mean():,.2f}")
print(f"🎯 Mediana de vendas: €{train['Sales'].median():,.2f}")
print(f"📊 Desvio padrão: €{train['Sales'].std():,.2f}")
print(f"🔼 Venda máxima registrada: €{train['Sales'].max():,.2f}")
print(f"\n👥 Média de clientes por dia: {train['Customers'].mean():.0f}")
print(f"🏪 Dias com loja aberta: {(train['Open'].sum()/len(train)*100):.1f}%")
print(f"🎁 Dias com promoção: {(train['Promo'].sum()/len(train)*100):.1f}%")

In [ ]:
# ============================================================
# 6️⃣ MERGE DOS DATASETS
# ============================================================
# Combinamos informações de vendas com características das lojas
# LEFT JOIN garante que mantemos todas as vendas, mesmo se houver
# lojas sem informações complementares

print("🔗 Realizando merge dos datasets...\n")

df = train.merge(store, on='Store', how='left')

print(f"✅ Shape após merge: {df.shape}")
print(f"📊 Colunas adicionadas: {df.shape[1] - train.shape[1]}")
print(f"\n📋 Novas colunas: {list(store.columns[1:])}")

# Verificar integridade do merge
print("\n🔍 Verificação de integridade:")
print(f"Registros originais: {len(train):,}")
print(f"Registros após merge: {len(df):,}")
print(f"Diferença: {len(df) - len(train)} ({'✅ OK' if len(df) == len(train) else '⚠️ REVISAR'})")

In [ ]:
# ============================================================
# 7️⃣ ANÁLISE EXPLORATÓRIA DE DADOS (EDA)
# ============================================================
# DISTRIBUIÇÃO DAS VENDAS
# Entender como as vendas estão distribuídas nos ajuda a:
# - Identificar outliers
# - Decidir sobre transformações (log, normalização)
# - Entender a natureza do problema

# Removemos vendas = 0 (lojas fechadas) para análise
df_open = df[df['Open'] == 1].copy()

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Distribuição de Vendas',
        'Boxplot - Detecção de Outliers',
        'Vendas por Dia da Semana',
        'Vendas: Com vs Sem Promoção'
    )
)

# Histograma
fig.add_trace(
    go.Histogram(x=df_open['Sales'], nbinsx=50, name='Vendas'),
    row=1, col=1
)

# Boxplot
fig.add_trace(
    go.Box(y=df_open['Sales'], name='Sales'),
    row=1, col=2
)

# Vendas por dia da semana
sales_by_day = df_open.groupby('DayOfWeek')['Sales'].mean().reset_index()
fig.add_trace(
    go.Bar(x=sales_by_day['DayOfWeek'], y=sales_by_day['Sales'], name='Média'),
    row=2, col=1
)

# Promoção
promo_impact = df_open.groupby('Promo')['Sales'].mean().reset_index()
fig.add_trace(
    go.Bar(x=['Sem Promo', 'Com Promo'], y=promo_impact['Sales'], 
           marker_color=['lightcoral', 'lightgreen']),
    row=2, col=2
)

fig.update_layout(height=800, showlegend=False, title_text="📊 Análise Exploratória de Vendas")
fig.show()

print("\n💡 OBSERVAÇÕES:")
print("="*60)
promo_lift = (promo_impact.loc[1, 'Sales'] / promo_impact.loc[0, 'Sales'] - 1) * 100
print(f"📈 Impacto da promoção: +{promo_lift:.1f}% nas vendas")
print(f"📅 Melhor dia da semana: {sales_by_day.loc[sales_by_day['Sales'].idxmax(), 'DayOfWeek']}")
print(f"📉 Pior dia da semana: {sales_by_day.loc[sales_by_day['Sales'].idxmin(), 'DayOfWeek']}")

In [ ]:
# ============================================================
# 8️⃣ ANÁLISE TEMPORAL - SÉRIES TEMPORAIS
# ============================================================
# Convertemos Date para datetime e analisamos tendências ao longo do tempo
# Isso é crucial para modelos de séries temporais

df['Date'] = pd.to_datetime(df['Date'])

# Vendas agregadas por dia
daily_sales = df.groupby('Date')['Sales'].sum().reset_index()

# Vendas por mês
df['YearMonth'] = df['Date'].dt.to_period('M')
monthly_sales = df.groupby('YearMonth')['Sales'].sum().reset_index()
monthly_sales['YearMonth'] = monthly_sales['YearMonth'].astype(str)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=daily_sales['Date'],
    y=daily_sales['Sales'],
    mode='lines',
    name='Vendas Diárias',
    line=dict(color='royalblue', width=1)
))

# Média móvel de 7 dias
daily_sales['MA7'] = daily_sales['Sales'].rolling(window=7).mean()
fig.add_trace(go.Scatter(
    x=daily_sales['Date'],
    y=daily_sales['MA7'],
    mode='lines',
    name='Média Móvel 7 dias',
    line=dict(color='red', width=2)
))

fig.update_layout(
    title='📈 Evolução Temporal das Vendas (2013-2015)',
    xaxis_title='Data',
    yaxis_title='Vendas Totais (€)',
    hovermode='x unified',
    height=500
)

fig.show()

print("\n💡 ANÁLISE TEMPORAL:")
print("="*60)
print(f"📆 Período analisado: {daily_sales['Date'].min().date()} a {daily_sales['Date'].max().date()}")
print(f"📊 Número de dias: {len(daily_sales)}")
print(f"📈 Tendência: {'Crescente' if daily_sales['Sales'].iloc[-30:].mean() > daily_sales['Sales'].iloc[:30].mean() else 'Decrescente'}")

In [ ]:
# ============================================================
# 9️⃣ FEATURE ENGINEERING - CRIAÇÃO DE VARIÁVEIS TEMPORAIS
# ============================================================
# Extraímos componentes da data que podem ser preditivos:
# - Year, Month, Day: sazonalidade
# - WeekOfYear: padrões semanais
# - IsWeekend: comportamento diferente em finais de semana
# - Quarter: trimestre do ano

print("🔧 Criando features temporais...\n")

df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['WeekOfYear'] = df['Date'].dt.isocalendar().week
df['DayOfYear'] = df['Date'].dt.dayofyear
df['Quarter'] = df['Date'].dt.quarter
df['IsWeekend'] = (df['DayOfWeek'] >= 6).astype(int)
df['IsMonthStart'] = df['Date'].dt.is_month_start.astype(int)
df['IsMonthEnd'] = df['Date'].dt.is_month_end.astype(int)

# Nome dos meses para análise
month_names = {1: 'Jan', 2: 'Fev', 3: 'Mar', 4: 'Abr', 5: 'Mai', 6: 'Jun',
               7: 'Jul', 8: 'Ago', 9: 'Set', 10: 'Out', 11: 'Nov', 12: 'Dez'}

print("✅ Features criadas:")
print("  - Year, Month, Day")
print("  - WeekOfYear, DayOfYear")
print("  - Quarter, IsWeekend")
print("  - IsMonthStart, IsMonthEnd")

# ADICIONE ESTA LINHA: Recriar df_open com as novas features
df_open = df[df['Open'] == 1].copy()

# Visualizar sazonalidade mensal
monthly_avg = df_open.groupby('Month')['Sales'].mean().reset_index()
monthly_avg['MonthName'] = monthly_avg['Month'].map(month_names)

fig = px.bar(monthly_avg, x='MonthName', y='Sales',
             title='📅 Sazonalidade: Vendas Médias por Mês',
             labels={'MonthName': 'Mês', 'Sales': 'Vendas Médias (€)'},
             color='Sales',
             color_continuous_scale='Blues')
fig.update_layout(showlegend=False)
fig.show()

print(f"\n📊 Mês com maior venda média: {monthly_avg.loc[monthly_avg['Sales'].idxmax(), 'MonthName']}")
print(f"📊 Mês com menor venda média: {monthly_avg.loc[monthly_avg['Sales'].idxmin(), 'MonthName']}")

In [ ]:
# ============================================================
# 🔟 FEATURE ENGINEERING - VARIÁVEIS DE LAG
# ============================================================
# Lag features capturam vendas passadas, essenciais para previsão
# - sales_lag_7: vendas de 7 dias atrás (mesma semana anterior)
# - sales_lag_14: vendas de 14 dias atrás (2 semanas)
# - sales_rolling_7: média móvel das últimas 7 vendas

print("🔧 Criando variáveis de lag (vendas passadas)...\n")

# Ordenar por loja e data
df = df.sort_values(['Store', 'Date']).reset_index(drop=True)

# Lags por loja
df['sales_lag_1'] = df.groupby('Store')['Sales'].shift(1)
df['sales_lag_7'] = df.groupby('Store')['Sales'].shift(7)
df['sales_lag_14'] = df.groupby('Store')['Sales'].shift(14)

# Médias móveis
df['sales_rolling_7'] = df.groupby('Store')['Sales'].transform(
    lambda x: x.shift(1).rolling(window=7, min_periods=1).mean()
)
df['sales_rolling_30'] = df.groupby('Store')['Sales'].transform(
    lambda x: x.shift(1).rolling(window=30, min_periods=1).mean()
)

# Tendência (diferença entre lag 7 e lag 14)
df['sales_trend'] = df['sales_lag_7'] - df['sales_lag_14']

print("✅ Features de lag criadas:")
print("  - sales_lag_1, sales_lag_7, sales_lag_14")
print("  - sales_rolling_7, sales_rolling_30")
print("  - sales_trend")

# Verificar valores ausentes criados pelos lags
print(f"\n⚠️ NAs criados (primeiros dias sem histórico): {df[['sales_lag_7', 'sales_lag_14']].isna().sum().max()}")

In [ ]:
# ============================================================
# 1️⃣1️⃣ ANÁLISE DE CORRELAÇÃO
# ============================================================
# Identificamos quais variáveis têm maior correlação com Sales
# Isso nos ajuda a entender o poder preditivo de cada feature

# Selecionar apenas colunas numéricas
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Remover YearMonth se existir (é categórica)
if 'YearMonth' in numeric_cols:
    numeric_cols.remove('YearMonth')

corr_matrix = df[numeric_cols].corr()

# Top 15 correlações com Sales
sales_corr = corr_matrix['Sales'].abs().sort_values(ascending=False)[1:16]

fig = px.bar(x=sales_corr.values, y=sales_corr.index, orientation='h',
             title='🔍 Top 15 Variáveis Correlacionadas com Vendas',
             labels={'x': 'Correlação Absoluta', 'y': 'Variável'},
             color=sales_corr.values,
             color_continuous_scale='RdYlGn')
fig.update_layout(showlegend=False, height=600)
fig.show()

print("\n💡 PRINCIPAIS CORRELAÇÕES:")
print("="*60)
for i, (var, corr) in enumerate(sales_corr.head(5).items(), 1):
    print(f"{i}. {var}: {corr:.3f}")

In [ ]:
# ============================================================
# 1️⃣2️⃣ TRATAMENTO DE VALORES AUSENTES E OUTLIERS
# ============================================================
# Estratégia para NAs:
# - CompetitionDistance: mediana (ausência = sem concorrência próxima)
# - CompetitionOpenSince: 0 (sem histórico de abertura)
# - Promo2: forward fill (manter última informação conhecida)
# - Lag features: remover linhas (primeiros dias sem histórico)

print("🧹 Tratando valores ausentes...\n")

# Competição
df['CompetitionDistance'].fillna(df['CompetitionDistance'].median(), inplace=True)
df['CompetitionOpenSinceMonth'].fillna(0, inplace=True)
df['CompetitionOpenSinceYear'].fillna(0, inplace=True)

# Promo2
df['Promo2SinceWeek'].fillna(0, inplace=True)
df['Promo2SinceYear'].fillna(0, inplace=True)
df['PromoInterval'].fillna('None', inplace=True)

# Remover linhas com NAs em lag features (primeiros 14 dias por loja)
df_clean = df.dropna(subset=['sales_lag_14']).copy()

print(f"📊 Linhas antes: {len(df):,}")
print(f"📊 Linhas após remoção de NAs: {len(df_clean):,}")
print(f"📉 Linhas removidas: {len(df) - len(df_clean):,} ({(1 - len(df_clean)/len(df))*100:.1f}%)")

# Verificação final de NAs
remaining_nas = df_clean.isnull().sum().sum()
print(f"\n✅ NAs restantes: {remaining_nas}")

In [ ]:
# ============================================================
# 1️⃣3️⃣ PREPARAÇÃO FINAL - SELEÇÃO DE FEATURES
# ============================================================
# Selecionamos features relevantes para modelagem
# Removemos: Date (já extraímos componentes), IDs temporários

print("🎯 Selecionando features para modelagem...\n")

# Features categóricas
cat_features = ['Store', 'DayOfWeek', 'Open', 'Promo', 'StateHoliday', 
                'SchoolHoliday', 'StoreType', 'Assortment', 'Promo2', 'PromoInterval']

# Features numéricas
num_features = ['CompetitionDistance', 'CompetitionOpenSinceMonth', 
                'CompetitionOpenSinceYear', 'Promo2SinceWeek', 'Promo2SinceYear',
                'Year', 'Month', 'Day', 'WeekOfYear', 'DayOfYear', 'Quarter',
                'IsWeekend', 'IsMonthStart', 'IsMonthEnd',
                'sales_lag_1', 'sales_lag_7', 'sales_lag_14',
                'sales_rolling_7', 'sales_rolling_30', 'sales_trend']

# Target
target = 'Sales'

# Todas as features
all_features = cat_features + num_features

print(f"📊 Total de features: {len(all_features)}")
print(f"   - Categóricas: {len(cat_features)}")
print(f"   - Numéricas: {len(num_features)}")
print(f"\n🎯 Target: {target}")

# Verificar se todas as features existem
missing_features = [f for f in all_features if f not in df_clean.columns]
if missing_features:
    print(f"\n⚠️ Features faltantes: {missing_features}")
else:
    print("\n✅ Todas as features estão disponíveis!")

In [ ]:
# ============================================================
# 1️⃣4️⃣ SPLIT TEMPORAL - TREINO E TESTE
# ============================================================
# Para séries temporais, usamos SPLIT TEMPORAL (não aleatório!)
# Últimos 20% dos dados como teste (aproximadamente 6 semanas)
# Isso simula uma previsão real: treinar no passado, prever o futuro

print("✂️ Realizando split temporal dos dados...\n")

# Ordenar por data
df_clean = df_clean.sort_values('Date').reset_index(drop=True)

# Definir ponto de corte (80% treino, 20% teste)
split_date = df_clean['Date'].quantile(0.8)

# Split
train_data = df_clean[df_clean['Date'] <= split_date].copy()
test_data = df_clean[df_clean['Date'] > split_date].copy()

print(f"📅 Data de corte: {split_date.date()}")
print(f"\n📊 TREINO:")
print(f"   Período: {train_data['Date'].min().date()} a {train_data['Date'].max().date()}")
print(f"   Registros: {len(train_data):,}")
print(f"\n📊 TESTE:")
print(f"   Período: {test_data['Date'].min().date()} a {test_data['Date'].max().date()}")
print(f"   Registros: {len(test_data):,}")
print(f"\n📈 Proporção: {len(train_data)/len(df_clean)*100:.1f}% treino / {len(test_data)/len(df_clean)*100:.1f}% teste")

# Separar X e y
X_train = train_data[all_features]
y_train = train_data[target]

X_test = test_data[all_features]
y_test = test_data[target]

print(f"\n✅ Shapes finais:")
print(f"   X_train: {X_train.shape}")
print(f"   y_train: {y_train.shape}")
print(f"   X_test: {X_test.shape}")
print(f"   y_test: {y_test.shape}")

In [ ]:
# ============================================================
# 1️⃣5️⃣ SALVAMENTO DOS DADOS PROCESSADOS
# ============================================================
# Salvamos os dados processados para uso no próximo notebook
# Isso evita reprocessar e garante consistência

import os

# Criar diretório se não existir
os.makedirs('../data/processed', exist_ok=True)

print("💾 Salvando dados processados...\n")

# Salvar datasets
train_data.to_csv('../data/processed/train_processed.csv', index=False)
test_data.to_csv('../data/processed/test_processed.csv', index=False)

# Salvar lista de features
import json
features_dict = {
    'all_features': all_features,
    'cat_features': cat_features,
    'num_features': num_features,
    'target': target
}

with open('../data/processed/features.json', 'w') as f:
    json.dump(features_dict, f, indent=2)

print("✅ Arquivos salvos:")
print("   - train_processed.csv")
print("   - test_processed.csv")
print("   - features.json")
print("\n📂 Localização: ../data/processed/")

In [ ]:
# ============================================================
# 1️⃣6️⃣ RESUMO FINAL
# ============================================================

print("="*70)
print("📋 RESUMO DO PROCESSAMENTO DE DADOS")
print("="*70)

print("\n✅ DADOS CARREGADOS:")
print(f"   - Train: {len(train):,} registros")
print(f"   - Store: {len(store)} lojas")
print(f"   - Período: {train['Date'].min()} a {train['Date'].max()}")

print("\n✅ TRANSFORMAÇÕES REALIZADAS:")
print("   ✓ Merge de datasets (vendas + características lojas)")
print("   ✓ Conversão de tipos de dados")
print("   ✓ Criação de 19 features temporais")
print("   ✓ Criação de 6 features de lag/tendência")
print("   ✓ Tratamento de valores ausentes")
print("   ✓ Split temporal (80/20)")

print("\n✅ DATASETS FINAIS:")
print(f"   - Treino: {len(train_data):,} registros ({len(X_train.columns)} features)")
print(f"   - Teste: {len(test_data):,} registros ({len(X_test.columns)} features)")

print("\n✅ ESTATÍSTICAS DO TARGET (TREINO):")
print(f"   - Média: €{y_train.mean():,.2f}")
print(f"   - Mediana: €{y_train.median():,.2f}")
print(f"   - Desvio padrão: €{y_train.std():,.2f}")
print(f"   - Mín/Máx: €{y_train.min():,.2f} / €{y_train.max():,.2f}")

print("\n🎯 PRÓXIMOS PASSOS:")
print("   1. Notebook 02: Modelagem e comparação de algoritmos")
print("   2. Treinar: Regressão Linear, Random Forest, XGBoost")
print("   3. Avaliar métricas: RMSE, MAE, MAPE, R²")
print("   4. Análise de importância de features")
print("   5. Otimização de hiperparâmetros")

print("\n" + "="*70)
print("✅ NOTEBOOK 01 CONCLUÍDO COM SUCESSO!")
print("="*70)